In [ ]:
#setup for processed data
#Note: use for BinaryArray data produced from Entrainment_Preprocessing.ipynb
def GetProcessedString(PROCESSING=False):
    if PROCESSING==True:
        Processed_string="PROCESSED_"
    else:
        Processed_string=""
    return Processed_string

#setup for dividing by massflux
def GetDivideMassFluxString(DivideMassFlux=False):
    if DivideMassFlux==True:
        DivideMassFlux_string="_DivideMassFlux"
    else:
        DivideMassFlux_string=""
    return DivideMassFlux_string
PROCESSED_DivideMassFlux_Options = [(False, False), (True, False), (True, True)]

#######################################################################################
OPTION=2
PROCESSING, DivideMassFlux = PROCESSED_DivideMassFlux_Options[OPTION]
Processed_string = GetProcessedString(PROCESSING=PROCESSING)
DivideMassFlux_string = GetDivideMassFluxString(DivideMassFlux=DivideMassFlux)

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import matplotlib.ticker as mticker
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

import warnings

#Additional Libraries
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter
from matplotlib.colors import TwoSlopeNorm
from scipy.ndimage import gaussian_filter
import matplotlib.patheffects as pe

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"Simulation_{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

sys.path.append(os.path.join(mainDirectory, "Code", "CodeFiles", "Functions", "Classes"))
from DataPlotting_Classes import DataPlotting_Classes
UltimatePlotting = DataPlotting_Classes.UltimatePlotting_Class

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=6)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation",dtype='float32')

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
#SBF location (cached by the tracking code, as in CloudTopFigure.py)
def LoadSBF_XLocations():
    DataManager_Tracking = DataManager_Class(mainDirectory, scratchDirectory, ModelData,
                                             dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                             dtype='float32', codeSection="Project_Algorithms")
    fileName = f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_{ModelData.Ntime}nt.pkl"
    with open(os.path.join(DataManager_Tracking.outputDataDirectory, fileName), "rb") as f:
        return pickle.load(f)

def IndexWithXMaxs(xh, xmaxs, removeOutliers=True):
    xmaxs = list(xmaxs).copy()
    xmaxs[0] = xmaxs[1]
    xmax_locations = xh[np.array(xmaxs).astype(int)].astype(float)
    xmax_locations[:np.abs(ModelData.time_hrs - 10).argmin()] = np.nan
    if removeOutliers:
        # remove values > 200 km between 10 and 11 LT (replace with the previous value)
        t_start = np.abs(ModelData.time_hrs - 10).argmin()
        t_end = np.abs(ModelData.time_hrs - 11).argmin()
        idx = np.arange(len(xmax_locations))
        mask = (idx >= t_start) & (idx <= t_end) & (xmax_locations > 200)
        for i in np.where(mask)[0]:
            xmax_locations[i] = xmax_locations[i - 1]
    return xmax_locations

In [ ]:
# #LFC and Cloudbase Calculation

# try:
#     #lfc
#     sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
#     from CLASSES_TrackingAlgorithms import Results_InputOutput_Class
#     def LoadLFCProfile():
#         DataManager_Tracking = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
#                                         ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
#                                         dtype='float32',codeSection = "Project_Algorithms")
#         LFC_profile = Results_InputOutput_Class.LoadLFC_Profile_Combined(ModelData,DataManager_Tracking, Ltype = "LFC")['LFC_profile']
#         LCL_profile = Results_InputOutput_Class.LoadLFC_Profile_Combined(ModelData,DataManager_Tracking, Ltype = "LCL")['LCL_profile']
#         return LFC_profile,LCL_profile
#     [LFC_profile,LCL_profile]=LoadLFCProfile()
#     MeanLFC = np.nanmean(LFC_profile)
#     MeanLCL = np.nanmean(LCL_profile)
#     print(f"Mean LFC is: {MeanLFC}\n");print(f"Mean LCL is: {MeanLCL}\n")
    
#     #cloudbase
#     def LoadAllCloudBase():
#         DataManager_Tracking = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
#                                         ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
#                                         dtype='float32',codeSection = "Project_Algorithms")
#         all_cloudbase = Results_InputOutput_Class.LoadAllCloudBase_Combined(ModelData,DataManager_Tracking)
#         return all_cloudbase['all_cloudbase']
#     min_all_cloudbase=np.nanmin(LoadAllCloudBase())
#     cloudbase=min_all_cloudbase
#     print(f"Minimum Cloudbase is: {cloudbase}\n")
# except:
#     MeanLFC=2
#     LFC_Profile = np.ones(ModelData.Ntime)*MeanLFC
#     cloudbase=1
    

In [ ]:
#Variable Loading Functions
def GetVariable_T(t, varName, cache=None): #*MassFlux
    """Load variable for a given timestep, using cached open HDF5 files if available."""
    if cache is None:
        cache = {}

    timeString = ModelData.timeStrings[t]

    # If file for this timestep already in cache, reuse it
    if timeString in cache:
        f = cache[timeString]
    else:
        # Otherwise, open new file and cache it
        f = DataManager.GetTimestepData_V2(DataManager.inputDataDirectory, timeString)
        cache[timeString] = f

    # Load the desired variable from the open file (lazy read)
    if varName in f.keys():
        output = f[varName][:]  # read actual data
    else:
        # fallback for derived variables
        output = CallVariable(ModelData, DataManager, timeString, variableName=varName)

    xh_mask = GetXMask()
    output = output.copy()
    output[..., ~xh_mask] = np.nan #removes data near the boundaries
    return output

def GetXMask():
    xh = ModelData.xh-ModelData.xh[0]
    left_frac, right_frac = 0.25, 0.80 #from coast to 20% of the right x-boundary
    return (xh >= left_frac*xh[-1]) & (xh <= right_frac*xh[-1])

def GetXPlotLimits():
    """Return the (min, max) of ModelData.xh within the kept 25-80% mask range."""
    xh = ModelData.xh
    xh_mask = GetXMask()
    return np.nanmin(xh[xh_mask]), np.nanmax(xh[xh_mask])

In [ ]:
#TAKING AVERAGES
    
def MeanOut_YX(output):
    output_mean = np.nanmean(output,axis=(1,2))
    return output_mean

def MeanOut_ZY(output):
    output_mean = np.nanmean(output,axis=(0,1))
    return output_mean

def MeanOut_ZX(output):
    output_mean = np.nanmean(output,axis=(0,2))
    return output_mean

def GetEntrainment_Mean(varNames, meantype='TZ'):
    #defining mean function
    if meantype=='TZ':
        MeanOut = MeanOut_YX
        dim1=ModelData.Ntime - 1
        dim2=ModelData.Nzh
    elif meantype=='TY':
        MeanOut = MeanOut_ZX
        dim1=ModelData.Ntime - 1
        dim2=ModelData.Nyh
    elif meantype=='TX': 
        MeanOut = MeanOut_ZY
        dim1=ModelData.Ntime - 1
        dim2=ModelData.Nxh
        
    outputDict = {name: np.zeros((dim1, dim2), dtype=np.float32) for name in varNames}
    loaded_cache = {} #cache for loaded datasets
    try:
        startTime=0; #startTime = np.abs(ModelData.time_hrs-10).argmin()
        for t in tqdm(range(startTime,ModelData.Ntime - 1)):
            # if np.mod(t,10)==0: print(f'Current time {t}')
    
            for varName in varNames:
                variable = GetVariable_T(t,varName,cache=loaded_cache)
                variableMean = MeanOut(variable)
                outputDict[varName][t] = variableMean

    finally:
        # Closing all h5py handles
        for key, f in loaded_cache.items():
            try:
                f.close()
            except Exception as e:
                print(f"Warning: failed to close file {key}: {e}")

    return outputDict


def GetEntrainment_MeanZX(varNames, meantype='ZX'):
    """
    Compute the mean over time and Y for each variable in varNames.
    Returns a dictionary with shape (zh, xh) for each variable.
    """
    outputDict = {name: np.zeros((ModelData.Nzh, ModelData.Nxh), dtype=np.float32) for name in varNames}
    countsDict = {name: np.zeros((ModelData.Nzh, ModelData.Nxh), dtype=np.float32) for name in varNames} #*MassFlux

    
    loaded_cache = {}
    try:
        startTime=0; # startTime = np.abs(ModelData.time_hrs-10).argmin()
        for t in tqdm(range(startTime,ModelData.Ntime - 1)):
            # if np.mod(t, 10) == 0: print(f'Current time {t}')

            for varName in varNames:
                variable = GetVariable_T(t, varName, cache=loaded_cache)
                # Average over y-axis (axis=2), result shape: (zh, xh)
                yz_mean = np.nanmean(variable, axis=1)

                # Accumulate over time
                # outputDict[varName] += yz_mean
                mask = ~np.isnan(yz_mean) #*MassFlux
                outputDict[varName][mask] += yz_mean[mask] #*MassFlux
                countsDict[varName][mask] += 1 #*MassFlux

        # Final average over time
        for varName in varNames:
            # nt = ModelData.Ntime - 1 
            # outputDict[varName] /= nt
            valid_mask = countsDict[varName] > 0 #*MassFlux
            outputDict[varName][valid_mask] /= countsDict[varName][valid_mask] #*MassFlux
            outputDict[varName][~valid_mask] = np.nan #*MassFlux

    finally:
        # Close all open HDF5 handles
        for key, f in loaded_cache.items():
            try:
                f.close()
            except Exception as e:
                print(f"Warning: failed to close file {key}: {e}")

    return outputDict

In [ ]:
def Calculate_EntrainmentDetrainment_Mean(meantype='TZ', forceCalculate=False):  
    if meantype=='ZX':
        GetEntrainment_Mean_Function = GetEntrainment_MeanZX
    else:
        GetEntrainment_Mean_Function = GetEntrainment_Mean
        
    #loading/calculation data
    try:
        if forceCalculate:
            raise RuntimeError("forceCalculate=True, skipping load")
        # Try loading existing calculations
        arrayDictionary = DataManager.LoadCalculations(
            DataManager.outputDataDirectory,
            dataName=f"{Processed_string}EntrainmentDetrainment{DivideMassFlux_string}_Mean{meantype}" #*MassFlux
        )
        print(arrayDictionary.keys())
    except Exception as e:
        print("Loading failed. Running calculations and saving.")
        print("Error:", e)

        varNames = [f"{Processed_string}Entrainment{DivideMassFlux_string}_g",
                    f"{Processed_string}Entrainment{DivideMassFlux_string}_c",
                    f"{Processed_string}Detrainment{DivideMassFlux_string}_g",
                    f"{Processed_string}Detrainment{DivideMassFlux_string}_c"]
        arrayDictionary = GetEntrainment_Mean_Function(varNames,meantype)
    
        #SAVING
        DataManager.SaveCalculations(DataManager.outputDataDirectory, arrayDictionary, dataName=f"{Processed_string}EntrainmentDetrainment{DivideMassFlux_string}_Mean{meantype}",dtype="float32") #*MassFlux

    a, b, c, d = (
        arrayDictionary[k] for k in [f"{Processed_string}Entrainment{DivideMassFlux_string}_g", 
                                     f"{Processed_string}Entrainment{DivideMassFlux_string}_c", 
                                     f"{Processed_string}Detrainment{DivideMassFlux_string}_g", 
                                     f"{Processed_string}Detrainment{DivideMassFlux_string}_c"]
    )
    return [a,b,c,d]

#entrainmentdetrainment

def Calculate_TransferEntrainmentDetrainment_Mean(meantype='TZ', forceCalculate=False):  
    if meantype=='ZX':
        GetEntrainment_Mean_Function = GetEntrainment_MeanZX
    else:
        GetEntrainment_Mean_Function = GetEntrainment_Mean
        
    #loading/calculation data
    try:
        if forceCalculate:
            raise RuntimeError("forceCalculate=True, skipping load")
        # Try loading existing calculations
        arrayDictionary = DataManager.LoadCalculations(
            DataManager.outputDataDirectory,
            dataName=f"{Processed_string}TransferEntrainmentDetrainment{DivideMassFlux_string}_Mean{meantype}" #*MassFlux
        )
        print(arrayDictionary.keys())
    except Exception as e:
        print("Loading failed. Running calculations and saving.")
        print("Error:", e)

        varNames = [f"{Processed_string}TransferEntrainment{DivideMassFlux_string}_g",
                    f"{Processed_string}TransferEntrainment{DivideMassFlux_string}_c",
                    f"{Processed_string}TransferDetrainment{DivideMassFlux_string}_g",
                    f"{Processed_string}TransferDetrainment{DivideMassFlux_string}_c"]
        arrayDictionary = GetEntrainment_Mean_Function(varNames,meantype)
    
        #SAVING
        DataManager.SaveCalculations(DataManager.outputDataDirectory, arrayDictionary, dataName=f"{Processed_string}TransferEntrainmentDetrainment{DivideMassFlux_string}_Mean{meantype}",dtype="float32") #*MassFlux

    a, b, c, d = (
        arrayDictionary[k] for k in [f"{Processed_string}TransferEntrainment{DivideMassFlux_string}_g", f"{Processed_string}TransferEntrainment{DivideMassFlux_string}_c", f"{Processed_string}TransferDetrainment{DivideMassFlux_string}_g", f"{Processed_string}TransferDetrainment{DivideMassFlux_string}_c"]
    )
    return [a,b,c,d]

In [ ]:
#Bulk Average Calculation Functions
def Calculate_EntrainmentDetrainment_BulkSums(forceCalculate=False):
    """Per-time sums of e, d, M over updraft boxes (M > 0), from the files NOT divided by M.
       TZ: summed over y, x -> (t, z);  TX: summed over y and z with dz weights -> (t, x)."""
    dataName = f"{Processed_string}EntrainmentDetrainment_BulkSums"
    try:
        if forceCalculate:
            raise RuntimeError("forceCalculate=True, skipping load")
        out = DataManager.LoadCalculations(DataManager.outputDataDirectory, dataName=dataName)
        print(out.keys())
        return out
    except Exception as e:
        print("Loading failed. Running calculations and saving.")
        print("Error:", e)

    Nt = ModelData.Ntime - 1
    dz = np.gradient(ModelData.zh)[:, None, None]
    out = {f'{k}_{sub}_Sum{kind}': np.zeros((Nt, ModelData.Nzh if kind == 'TZ' else ModelData.Nxh), dtype=np.float32)
           for sub in ('g', 'c') for k in ('e', 'd', 'M') for kind in ('TZ', 'TX')}

    cache = {}
    BULK_M_NAME = "MassFlux_{sub}" 
    try:
        for t in tqdm(range(Nt)):
            for sub in ('g', 'c'):
                e = GetVariable_T(t, f"{Processed_string}Entrainment_{sub}", cache=cache)
                d = GetVariable_T(t, f"{Processed_string}Detrainment_{sub}", cache=cache)
                M = GetVariable_T(t, BULK_M_NAME.format(sub=sub), cache=cache)
                valid = np.isfinite(M) & (M > 0)
                for k, F in (('e', e), ('d', d), ('M', M)):
                    F = np.where(valid & np.isfinite(F), F, 0.0)
                    out[f'{k}_{sub}_SumTZ'][t] = F.sum(axis=(1, 2))
                    out[f'{k}_{sub}_SumTX'][t] = (F * dz).sum(axis=(0, 1))
    finally:
        for f in cache.values():
            try: f.close()
            except Exception: pass

    DataManager.SaveCalculations(DataManager.outputDataDirectory, out, dataName=dataName, dtype="float32")
    return out

def BulkRatio(num, den):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(den > 0, num / den, np.nan)

In [ ]:
####################################
#CALCULATION
running=True #keep True when running job_array
running=False 

In [ ]:
#TZ Mean Calculations
[profile_array_e_g_MeanTZ,
 profile_array_e_c_MeanTZ,
 profile_array_d_g_MeanTZ,
 profile_array_d_c_MeanTZ] = Calculate_EntrainmentDetrainment_Mean(meantype='TZ')

#transfer entrainment calculations
[TransferEntrainment_g_MeanTZ,
 TransferEntrainment_c_MeanTZ,
 TransferDetrainment_g_MeanTZ,
 TransferDetrainment_c_MeanTZ] = Calculate_TransferEntrainmentDetrainment_Mean(meantype='TZ')

if running:
    del profile_array_e_g_MeanTZ,profile_array_e_c_MeanTZ,profile_array_d_g_MeanTZ,profile_array_d_c_MeanTZ
    del TransferEntrainment_g_MeanTZ,TransferEntrainment_c_MeanTZ,TransferDetrainment_g_MeanTZ,TransferDetrainment_c_MeanTZ

In [ ]:
#TY Mean Calculations
[profile_array_e_g_MeanTY,
 profile_array_e_c_MeanTY,
 profile_array_d_g_MeanTY,
 profile_array_d_c_MeanTY] = Calculate_EntrainmentDetrainment_Mean(meantype='TY')

if running:
    del profile_array_e_g_MeanTY,profile_array_e_c_MeanTY,profile_array_d_g_MeanTY,profile_array_d_c_MeanTY

In [ ]:
#TX Mean Calculations
[profile_array_e_g_MeanTX,
 profile_array_e_c_MeanTX,
 profile_array_d_g_MeanTX,
 profile_array_d_c_MeanTX] = Calculate_EntrainmentDetrainment_Mean(meantype='TX')

if running:
    del profile_array_e_g_MeanTX,profile_array_e_c_MeanTX,profile_array_d_g_MeanTX,profile_array_d_c_MeanTX

In [ ]:
#ZX Mean Calculations
[profile_array_e_g_MeanZX,
 profile_array_e_c_MeanZX,
 profile_array_d_g_MeanZX,
 profile_array_d_c_MeanZX] = Calculate_EntrainmentDetrainment_Mean(meantype='ZX')

if running:
    del profile_array_e_g_MeanZX,profile_array_e_c_MeanZX,profile_array_d_g_MeanZX,profile_array_d_c_MeanZX

In [ ]:
#Bulk Average Calculation
BulkSums = Calculate_EntrainmentDetrainment_BulkSums()

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
#Plot Calculation Functions

#Mean Functions
def GetData_Mean(type,meantype):
    if type=='general':
        profile_array_e=globals()[f"profile_array_e_g_Mean{meantype}"]
        profile_array_d=-globals()[f"profile_array_d_g_Mean{meantype}"]
        profile_array_net=profile_array_e-profile_array_d
    if type=='cloudy':
        profile_array_e=globals()[f"profile_array_e_c_Mean{meantype}"]
        profile_array_d=-globals()[f"profile_array_d_c_Mean{meantype}"]
        profile_array_net=profile_array_e-profile_array_d
    return profile_array_e,profile_array_d,profile_array_net

def GetTransferData_Mean(meantype='TZ'):
    TransferEntrainment_g_Mean=globals()[f"TransferEntrainment_g_Mean{meantype}"]
    TransferEntrainment_c_Mean=globals()[f"TransferEntrainment_c_Mean{meantype}"]
    TransferDetrainment_g_Mean=globals()[f"TransferDetrainment_g_Mean{meantype}"]
    TransferDetrainment_c_Mean=globals()[f"TransferDetrainment_c_Mean{meantype}"]
    return TransferEntrainment_g_Mean,TransferEntrainment_c_Mean, TransferDetrainment_g_Mean,TransferDetrainment_c_Mean

def GetBulkTimeMeanProfile(type):
    """Bulk time-mean profiles: sum e and d over all times, divide by sum of M (m^-1).
       Returns eps and delta (delta positive), matching GetData_Mean."""
    sub = 'g' if type == 'general' else 'c'
    e = BulkSums[f'e_{sub}_SumTZ'].sum(axis=0)
    d = BulkSums[f'd_{sub}_SumTZ'].sum(axis=0)
    M = BulkSums[f'M_{sub}_SumTZ'].sum(axis=0)
    return BulkRatio(e, M), -BulkRatio(d, M)

# bulk eps, delta arrays (m^-1), named so GetData_Mean(meantype='BulkTZ'/'BulkTX') finds them
for sub in ('g', 'c'):
    for kind in ('TZ', 'TX'):
        M = BulkSums[f'M_{sub}_Sum{kind}']
        globals()[f"profile_array_e_{sub}_MeanBulk{kind}"] = BulkRatio(BulkSums[f'e_{sub}_Sum{kind}'], M)
        globals()[f"profile_array_d_{sub}_MeanBulk{kind}"] = BulkRatio(BulkSums[f'd_{sub}_Sum{kind}'], M)

In [ ]:
#Contour and Profile Plotting Functions
def Plot_All_Mean_Entrainment(fontScale=1.4, figSize=(16, 15), tickSize=16,
                              xKmLim=(125, 400), xKmStep=25,
                              tLim=(9, 17), tStep=1,
                              ratioContours=True, average='simple'):   # 'simple' or 'bulk'
    UP = UltimatePlotting
    F = DataPlotting_Classes.Functions
    EPS_TO_PER_KM = 1000.0 if DivideMassFlux else 1.0   # stored eps, delta are in m^-1 -> km^-1
    meanTZ, meanTX = ('BulkTZ', 'BulkTX') if average == 'bulk' else ('TZ', 'TX')
    [fs] = UP.GetFontStyles(fontScale)
    units = r"$\mathrm{km^{-1}}$" if DivideMassFlux else r"$\mathrm{kg\ m^{-3}\ s^{-1}}$"
    title = lambda ax, s: ax.set_title(s, fontsize=fs["title"]["fontsize"], fontweight='normal')
    letter = lambda r, c: "ABCDEF"[2 * r + c]

    [fig, axes, caxes] = UP.MakeAlignedGridFigure(
        nRows=3, nCols=2, figSize=figSize,
        hasCbar=[[True, False], [True, False], [True, True]],
        cbarSide="right", rowGapRatio=0.20, colGapRatio=0.2,
        cbarPaddingRatioRight=0.02, cbarThicknessRatio=0.04)

    # ---- sharing (set before plotting) ----
    axes[1, 0].sharex(axes[0, 0])                # A and C share time
    axes[2, 1].sharex(axes[2, 0])                # E and F share time
    for r in (0, 1):
        axes[r, 1].sharey(axes[r, 0])            # profile shares z with its contour
    axes[2, 1].sharey(axes[2, 0])                # E and F share x (km)

    xh_rel = ModelData.xh - ModelData.xh[0]
    sbf = IndexWithXMaxs(xh_rel, LoadSBF_XLocations())
    xLim = xKmLim
    xTicksKm = np.arange(xKmLim[0], xKmLim[1] + 1e-6, xKmStep)
    zh = ModelData.zh
    tTicks = np.arange(tLim[0], tLim[1] + 1e-6, tStep)
    ratio = GetCloudyEntrainmentRatio() if ratioContours else None

    cases = [('general', 'general', (0, 3), 'g'),
             ('cloudy',  'cloudy',  (0, 16), 'c')]

    # ================= rows 1-2 (A-D): z-t contour | mean profile =================
    for r, (typ, name, zLim, sub) in enumerate(cases):
        S = GetEntrainmentSymbols(sub)
        e, d, n = (EPS_TO_PER_KM * a for a in GetData_Mean(type=typ, meantype=meanTZ))
        tt = ModelData.time_hrs[:n.shape[0]]
        zStep = 0.5 if (zLim[1] - zLim[0]) <= 3 else 2
        zTicks = np.arange(zLim[0], zLim[1] + 1e-6, zStep)

        levels, norm, ticks = NetEntrainmentColorSettings(n)
        [_, ax, cbar] = UP.PlotContour(
            ax=axes[r, 0], cax=caxes[r, 0], xData=tt, yData=zh, zData=n.T,
            cmap='seismic_r', levels=levels, norm=norm, extend='both',
            cbarTicks=ticks, cbarLabel=None, yLabel='z (km)',
            xLabel=None,
            xTicks=tTicks, xLim=tLim, yTicks=zTicks, yLim=zLim,
            fontScale=fontScale)
        ax.set_facecolor('gray')
        if not DivideMassFlux:
            cbar.formatter.set_powerlimits((0, 0)); cbar.update_ticks()
        title(ax, f"{letter(r, 0)}) {S['net']} ({units})")
        ax.set_xlabel('')
        plt.setp(ax.get_xticklabels(), visible=False)

        if ratioContours:
            AddCloudyEntrainmentRatioContours(ax, tt, zh, ratio)
            ax.set_xlim(tLim); ax.set_ylim(zLim)

        ax = axes[r, 1]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            if average == 'bulk':
                em, dm = (EPS_TO_PER_KM * a for a in GetBulkTimeMeanProfile(typ))
            else:
                em, dm = np.nanmean(e, axis=0), np.nanmean(d, axis=0)
        for y, c, lab in [(em, 'blue', S['e']), (dm, 'red', S['d']), (em - dm, 'black', S['net'])]:
            UP.PlotLine(ax=ax, xData=y, yData=zh, color=c, label=lab, showLegend=False, fontScale=fontScale)
        ax.axvline(0, color='gray', ls='--', zorder=-10)
        ax.set_ylim(zLim); ax.set_yticks(zTicks); ax.grid(True)
        plt.setp(ax.get_yticklabels(), visible=False)
        zmask = (zh >= zLim[0]) & (zh <= zLim[1])
        SetNiceValueAxis(ax, [em[zmask], dm[zmask], (em - dm)[zmask]], axis='x')
        if not DivideMassFlux:
            F.ApplyScientificNotation_axes([ax], dimension='x', decimals=1)
        ax.legend(loc='upper right', fontsize=fs["legend"]["fontsize"])
        title(ax, f"{letter(r, 1)}) Mean profile ({name})")

    # ================= row 3 (E-F): x-t contour =================
    for j, (typ, name, zLim, sub) in enumerate(cases):
        S = GetEntrainmentSymbols(sub)
        e, d, n = (EPS_TO_PER_KM * a for a in GetData_Mean(type=typ, meantype=meanTX))
        tt = ModelData.time_hrs[:n.shape[0]]
        levels, norm, ticks = NetEntrainmentColorSettings(n)
        [_, ax, cbar] = UP.PlotContour(
            ax=axes[2, j], cax=caxes[2, j], xData=tt, yData=xh_rel, zData=n.T,
            cmap='seismic_r', levels=levels, norm=norm, extend='both',
            cbarTicks=ticks, cbarLabel=None, yLabel='x (km)' if j == 0 else None,
            xLabel='Local time (LT)',
            xTicks=tTicks, xLim=tLim, yTicks=xTicksKm, yLim=xLim,
            fontScale=fontScale)
        ax.set_facecolor('gray')
        if not DivideMassFlux:
            cbar.formatter.set_powerlimits((0, 0)); cbar.update_ticks()
        title(ax, f"{letter(2, j)}) {S['net']} ({units})")
        ax.plot(ModelData.time_hrs, sbf, color='black', lw=3, zorder=20, scalex=False, scaley=False,
                path_effects=[pe.Stroke(linewidth=5, foreground='white'), pe.Normal()])
        if j == 1: plt.setp(ax.get_yticklabels(), visible=False)

    axes[2, 0].set_ylim(xKmLim); axes[2, 0].set_yticks(xTicksKm)
    axes[2, 0].set_xlim(tLim)

    if tickSize is not None:
        for ax in axes.ravel():
            ax.tick_params(axis='both', which='major', labelsize=tickSize)
    return fig

def GetEntrainmentSymbols(sub):
    """sub = 'g' (general) or 'c' (cloudy). E/D when not divided by mass flux, epsilon/delta when it is."""
    if DivideMassFlux:
        return dict(e=rf'$\epsilon_{{{sub}}}$', d=rf'$\delta_{{{sub}}}$',
                    net=rf'$\epsilon_{{{sub}}} - \delta_{{{sub}}}$')
    return dict(e=rf'$E_{{{sub}}}$', d=rf'$D_{{{sub}}}$',
                net=rf'$E_{{{sub}}} - D_{{{sub}}}$')

def NetEntrainmentColorSettings(n, n_levels=29):
    vmin, vmax = np.nanpercentile(n, 1), np.nanpercentile(n, 99)
    [levels] = BuildZeroCenteredLevels(vmin, vmax, n_levels)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    ticks = levels[::max(1, len(levels) // 4)]
    return levels, norm, ticks

def BuildZeroCenteredLevels(vMin, vMax, nLevels):
    # Equal number of levels on each side of zero, regardless of how
    # asymmetric vMin/vMax are. Level *spacing* differs per side instead.
    numPerSide = max(2, nLevels // 2)

    negativeLevels = np.linspace(vMin, 0, numPerSide + 1)   # includes 0
    positiveLevels = np.linspace(0, vMax, numPerSide + 1)   # includes 0
    levels = np.unique(np.concatenate([negativeLevels, positiveLevels]))
    return [levels]

def SetNiceValueAxis(ax, series, axis='x', nTicks=5):
    """Nice ticks and limits from the plotted series. Zero is kept inside the range."""
    vals = [np.asarray(s, float) for s in series] + [np.array([0.0])]
    [ticks, lim, _] = UltimatePlotting.GenerateTicksAndLim(vals, nTicks=nTicks)
    if axis == 'x':
        ax.set_xticks(ticks); ax.set_xlim(lim)
    else:
        ax.set_yticks(ticks); ax.set_ylim(lim)

def GetHeightIndexRange(minHeight, maxHeight):
    startIndex = np.searchsorted(ModelData.zh, minHeight, side='left')
    endIndex = np.searchsorted(ModelData.zh, maxHeight, side='right')
    return [startIndex, endIndex]

def GetCloudyEntrainmentRatio(sigma=(3, 1), minFrac=1e-3):
    """% of cloudy-updraft entrainment from general updrafts (eps_{g->c}/eps_c) on the TZ grid.
    Numerator and denominator are smoothed separately (NaN -> 0), then divided."""
    _, TE_c, _, _ = GetTransferData_Mean('TZ')
    e_c, _, _ = GetData_Mean(type='cloudy', meantype='TZ')
    num = gaussian_filter(np.nan_to_num(np.abs(TE_c)), sigma)
    den = gaussian_filter(np.nan_to_num(np.abs(e_c)), sigma)
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(den > minFrac * np.nanmax(den), 100 * num / den, np.nan)

def AddCloudyEntrainmentRatioContours(ax, tt, zh, ratio, levels=(25, 50, 75),
                                      linestyles=('dotted','solid', 'dashed'), color='black', lw=1.5):
    """One black contour per level; NaN -> 0 for drawing so contours close at the cloudy-entrainment edge."""
    ratio_plot = np.nan_to_num(ratio[:len(tt)], nan=0.0)
    for lev, ls in zip(levels, linestyles):
        ax.contour(tt, zh, ratio_plot.T, levels=[lev], colors=color,
                   linewidths=lw, linestyles=ls, zorder=30)

plt.rcParams['figure.dpi'] = 300

In [ ]:
####################################
#PLOTTING

In [ ]:
#Contours and Profiles
if not running:
    fig = Plot_All_Mean_Entrainment(average='bulk') # average='simple'
    
    #SAVING 
    fileName=f"{Processed_string}EntrainmentDetrainment{DivideMassFlux_string}_Mean" 
    SaveFigure(fig,plotType="Variable_Calculation/EntrainmentCalculation",fileName=fileName)